In [0]:
%sql

-- Group: fk_analysts
-- Read-only access to gold schema only

GRANT USE CATALOG ON CATALOG fk_t12 TO `fk_analyst`;

GRANT USE SCHEMA ON SCHEMA fk_t12.gold TO `fk_analyst`;
GRANT SELECT ON SCHEMA fk_t12.gold TO `fk_analyst`;



-- Group: fk_engineers
-- Full read/write across bronze, silver, gold, ops
-- Plus volume read/write on landing.raw

GRANT USE CATALOG ON CATALOG fk_t12 TO `fk_engineers`;

-- Schema-level USE + SELECT + MODIFY on all four schemas
GRANT USE SCHEMA ON SCHEMA fk_t12.bronze TO `fk_engineers`;
GRANT SELECT     ON SCHEMA fk_t12.bronze TO `fk_engineers`;
GRANT MODIFY     ON SCHEMA fk_t12.bronze TO `fk_engineers`;

GRANT USE SCHEMA ON SCHEMA fk_t12.silver TO `fk_engineers`;
GRANT SELECT     ON SCHEMA fk_t12.silver TO `fk_engineers`;
GRANT MODIFY     ON SCHEMA fk_t12.silver TO `fk_engineers`;

GRANT USE SCHEMA ON SCHEMA fk_t12.gold TO `fk_engineers`;
GRANT SELECT     ON SCHEMA fk_t12.gold TO `fk_engineers`;
GRANT MODIFY     ON SCHEMA fk_t12.gold TO `fk_engineers`;

GRANT USE SCHEMA ON SCHEMA fk_t12.ops TO `fk_engineers`;
GRANT SELECT     ON SCHEMA fk_t12.ops TO `fk_engineers`;
GRANT MODIFY     ON SCHEMA fk_t12.ops TO `fk_engineers`;

-- Volume-level READ/WRITE on landing.raw
GRANT READ VOLUME  ON VOLUME fk_t12.landing.raw TO `fk_engineers`;
GRANT WRITE VOLUME ON VOLUME fk_t12.landing.raw TO `fk_engineers`;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS fk_t12.ops.user_entitlements (
  user_email    STRING,
  allowed_city  STRING,
  can_view_pii  BOOLEAN
);

In [0]:
%sql
CREATE OR REPLACE FUNCTION fk_t12.ops.mask_email(email_col STRING)
RETURNS STRING
RETURN
  CASE
    WHEN is_account_group_member("fk_admins") THEN email_col
    WHEN EXISTS (
        SELECT 1 FROM fk_t12.ops.user_entitlements e
        WHERE lower(e.user_email) = lower(current_user())
        AND e.can_view_pii = true
        ) THEN email_col
    ELSE
        CONCAT(
          SUBSTR(email_col, 1, 2),
          REPEAT('*', GREATEST(LENGTH(SUBSTR(email_col, 1, INSTR(email_col, '@') - 1)) - 2, 1)),
          SUBSTR(email_col, INSTR(email_col, '@'))
        )
  END;

CREATE OR REPLACE FUNCTION fk_t12.ops.mask_phone(phone_col STRING)
RETURNS STRING
RETURN
  CASE
    WHEN is_account_group_member("fk_admins") THEN phone_col
    WHEN EXISTS (
        SELECT 1 FROM fk_t12.ops.user_entitlements e
        WHERE lower(e.user_email) = lower(current_user())
        AND e.can_view_pii = true
        ) THEN phone_col
    ELSE
        CONCAT(
          SUBSTR(phone_col, 1, 2),
          REPEAT('*', GREATEST(LENGTH(phone_col) - 4, 1)),
          SUBSTR(phone_col, -2)
        )
  END;

ALTER TABLE fk_t12.gold.dim_customer
    ALTER COLUMN email SET MASK fk_t12.ops.mask_email;

ALTER TABLE fk_t12.gold.dim_customer
    ALTER COLUMN phone SET MASK fk_t12.ops.mask_phone;

In [0]:
%sql

CREATE OR REPLACE FUNCTION fk_t12.ops.store_sales_row_filter(city_col STRING)
RETURNS BOOLEAN
RETURN
  is_account_group_member('fk_admins')
  OR EXISTS (
    SELECT 1 FROM fk_t12.ops.user_entitlements e
    WHERE lower(e.user_email) = lower(current_user())
      AND (e.allowed_city = city_col OR e.allowed_city = 'ALL')
  );


ALTER TABLE fk_t12.gold.agg_daily_store_sales
  SET ROW FILTER fk_t12.ops.store_sales_row_filter ON (city);

In [0]:
%sql

CREATE TABLE IF NOT EXISTS fk_t12.ops.governance_evidence (
  evidence_name   STRING,
  observed_value  STRING
) USING DELTA;

-- merging the value so indempotent can support

MERGE INTO fk_t12.ops.user_entitlements AS t
USING (SELECT
    current_user() AS user_email,
    "Jaipur" AS allowed_city,
    FALSE AS can_view_pii
  ) AS s
ON lower(t.user_email) = lower(s.user_email)
WHEN MATCHED THEN
    UPDATE SET t.allowed_city = s.allowed_city, t.can_view_pii = s.can_view_pii
WHEN NOT MATCHED THEN
    INSERT *;

In [0]:
%sql
-- SELECT * FROM fk_t12.ops.user_entitlements WHERE lower(user_email) = lower(current_user());

-- Row count you see in gold.agg_daily_store_sales (post row filter)
INSERT INTO fk_t12.ops.governance_evidence
(evidence_name , observed_value)
SELECT 'ROW_FILTER_ROW_COUNT_JAIPUR', Cast(count(*) as string)
FROM fk_t12.gold.agg_daily_store_sales;

INSERT INTO fk_t12.ops.governance_evidence
(evidence_name , observed_value)
SELECT 'MASKED_EMAIL_C0001', email
FROM fk_t12.gold.dim_customer
WHERE customer_id = "C0001";

INSERT INTO fk_t12.ops.governance_evidence
(evidence_name , observed_value)
SELECT 'MASKED_PHONE_C0001', phone
FROM fk_t12.gold.dim_customer
WHERE customer_id = "C0001";
